# SHAKENBAYEV ML MIDTERM PROJECT

In [1]:
import sys, sklearn, pandas as pd, numpy as np

SEED = 3246

print(sys.version)
print("sklearn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
print(train.shape, test.shape)

3.13.9 (tags/v3.13.9:8183fa5, Oct 14 2025, 14:09:13) [MSC v.1944 64 bit (AMD64)]
sklearn 1.7.2 | pandas 2.3.3 | numpy 2.3.4
(32940, 22) (8236, 20)


## Stage 1 · Frame the problem 

In [2]:
print("Колонки только в train:", set(train.columns) - set(test.columns))
print("Колонки только в test: ", set(test.columns) - set(train.columns))
print()
print(train.columns.tolist())

Колонки только в train: {'duration', 'y'}
Колонки только в test:  set()

['id', 'age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']


# Stage 1 · Framing the problem

## What is predicted, for whom, and why 10 % matters

**Target.** For each client of a Portuguese bank we predict whether they will say *yes* (`y = yes`) to a term-deposit offer made by phone. The model is used *before* any call is made, to score a list of clients.

**Constraint.** The call centre can phone only 10 % of the clients on a list in one campaign wave. So the model is not asked for a yes/no verdict on every client; it must produce a **score that ranks clients**, so that the top 10 % contain as many future "yes" clients as possible.

**Metric.** Accuracy is not suitable: the target is imbalanced (about 11.3 % "yes"), so a classifier that always answers "no" gets about 88.7 % accuracy and finds nobody. Since only the ranking matters, I will:
- select and compare models by **ROC-AUC** and **average precision** (both depend only on the ranking and are computed per CV fold, with mean and std);
- report the business result with **precision and recall in the top 10 %** and the **lift** over calling at random.

## Train vs. test columns

`train.csv` has two columns that `test.csv` does not have:
- **`y`** — the answer we predict. The instructor keeps it hidden to score the model.
- **`duration`** — the length of the call in seconds.

`duration` is known only *after* the call has happened, and it is strongly tied to the outcome (a long conversation usually means an interested client). At scoring time, before any call, it does not exist. A model trained with it would look very good in cross-validation but could not be used in practice (and cannot run on `test.csv`). This is **target leakage**.

**Consequence for the model:** I may only use information available before the call. I drop `duration` from the features. I also drop `id`, because it is only a row number and not a property of the client.